<a href="https://colab.research.google.com/github/Leonardozepeda04/etl-data-pipeline2509832022/blob/main/notebooks/C_pagos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [3]:
#URL csv Aseguradoras
url_pagos= "https://raw.githubusercontent.com/Leonardozepeda04/etl-data-pipeline2509832022/refs/heads/main/data/raw/C_pagos.csv"

In [4]:
# Cargar archivo
pagos = pd.read_csv(url_pagos)

In [5]:
pagos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 246 entries, 0 to 245
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id_pago       246 non-null    object
 1   id_venta      239 non-null    object
 2   metodo        246 non-null    object
 3   monto_pagado  236 non-null    object
dtypes: object(4)
memory usage: 7.8+ KB


In [6]:
# Limpieza general
def limpiar_dataframe(df):
    df = df.copy()

    # Limpiar nombres de columnas (convertir a minúsculas y quitar espacios)
    df.columns = df.columns.str.strip().str.lower()

    # Eliminar espacios en blanco de todas las columnas de tipo texto
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip()

    # Reemplazar valores vacíos con NaN
    df = df.replace(r'^\s*$', pd.NA, regex=True)
    df = df.replace(['error', 'text_43'], pd.NA)  # Valores erróneos específicos

    # Eliminar duplicados
    df = df.drop_duplicates()

    return df

In [7]:
# Aplicar limpieza general
pagos = limpiar_dataframe(pagos)

In [8]:
# Transformaciones Importantes

# 1. Normalizar método de pago (capitalizar)
pagos['metodo'] = pagos['metodo'].astype(str).str.strip().str.capitalize()

# 2. Convertir monto_pagado a formato numérico
pagos['monto_pagado'] = pagos['monto_pagado'].apply(lambda x: str(x).replace(',', '').replace(' ', '').replace('$', '').strip())
pagos['monto_pagado'] = pd.to_numeric(pagos['monto_pagado'], errors='coerce')

# 3. Rellenar valores nulos en monto_pagado con 0
pagos['monto_pagado'] = pagos['monto_pagado'].fillna(0)

# 4. Rellenar valores nulos en id_venta con un valor predeterminado o NaN
pagos['id_venta'] = pagos['id_venta'].replace([None, '', 'nan'], pd.NA)

In [9]:
#Verificar resultados
print(pagos.head(15))

   id_pago id_venta         metodo  monto_pagado
0   PG8000    V9057  Transferencia       4452.68
1   PG8001    V9027  Transferencia       1742.47
2   PG8002    V9025        Tarjeta        789.21
3   PG8003    V9112  Transferencia        146.73
4   PG8004    V9190       Efectivo       3236.20
5   PG8005    V9065         Cheque       3191.77
6   PG8006    V9152         Cheque       1716.21
7   PG8007    V9130     Pago móvil        377.13
8   PG8008    V9199       Efectivo       2059.95
9   PG8009    V9043  Transferencia       1569.86
10  PG8010    V9096         Cheque       3392.06
11  PG8011    V9041       Efectivo       3036.78
12  PG8012    V9226         Cheque       1835.68
13  PG8013    V9005  Transferencia       1105.10
14  PG8014    V9166         Cheque       4940.42


In [10]:
# Separar válidos y rechazados
validos = pagos[
    pagos['id_pago'].notna() &
    pagos['id_venta'].notna() &
    pagos['metodo'].notna() &
    pagos['monto_pagado'].notna()
].copy()

rechazados = pagos[
    pagos['id_pago'].isna() |
    pagos['id_venta'].isna() |
    pagos['metodo'].isna() |
    pagos['monto_pagado'].isna()
].copy()

In [11]:
# Motivos de rechazo
def motivo_rechazo(row):
    motivos = []
    if pd.isna(row['id_pago']):
        motivos.append("id_pago_vacio")
    if pd.isna(row['id_venta']):
        motivos.append("id_venta_vacio")
    if pd.isna(row['metodo']):
        motivos.append("metodo_vacio")
    if pd.isna(row['monto_pagado']):
        motivos.append("monto_pagado_vacio")
    return ",".join(motivos)

In [12]:
# Aplicar la función para obtener los motivos de rechazo
rechazados["motivo_rechazo"] = rechazados.apply(motivo_rechazo, axis=1)

In [13]:
# Ver los primeros registros después de las transformaciones
print("Registros válidos:")
print(validos.head())

print("Registros rechazados:")
print(rechazados[['id_pago', 'id_venta', 'metodo', 'monto_pagado', 'motivo_rechazo']].head())

Registros válidos:
  id_pago id_venta         metodo  monto_pagado
0  PG8000    V9057  Transferencia       4452.68
1  PG8001    V9027  Transferencia       1742.47
2  PG8002    V9025        Tarjeta        789.21
3  PG8003    V9112  Transferencia        146.73
4  PG8004    V9190       Efectivo       3236.20
Registros rechazados:
    id_pago id_venta         metodo  monto_pagado  motivo_rechazo
29   PG8029     <NA>         Cheque       2303.44  id_venta_vacio
38   PG8038     <NA>       Efectivo       4920.39  id_venta_vacio
42   PG8042     <NA>        Tarjeta       1939.85  id_venta_vacio
52   PG8052     <NA>     Pago móvil       1311.49  id_venta_vacio
133  PG8133     <NA>  Transferencia       4788.09  id_venta_vacio


In [14]:
# Exportar archivos

validos.to_csv("pagos_validos.csv", index=False)
rechazados.to_csv("pagos_rejects.csv", index=False)
pagos.to_csv("pagos_curated.csv", index=False)